# บทที่ 8 การลดมิติและตัวแปรรวมทางฟิสิกส์
โน้ตบุ๊กประกอบตำรา *วิทยาศาสตร์ข้อมูลและการเรียนรู้ของเครื่องสำหรับนักฟิสิกส์*
โค้ดทุกเซลล์คัดลอกมาจากสคริปต์ในโฟลเดอร์ `ch08_dimred/` หัวเซลล์ **โค้ดที่ N.M** ตรงกับเลขในตำรา

**วิธีใช้**
1. รันเซลล์ **เตรียมโค้ด** ด้านล่างก่อนเสมอ (ทุกครั้งที่เปิดโน้ตบุ๊กหรือเริ่ม session ใหม่)
2. แต่ละหัวข้อเริ่มด้วยเซลล์ที่ขึ้นต้นด้วย `%reset -f` ซึ่งล้างตัวแปรเก่า ทำให้ผลเหมือนรันสคริปต์นั้นเดี่ยว ๆ
   ให้รันเซลล์ในหัวข้อเดียวกันเรียงจากบนลงล่าง (Shift+Enter) หรือใช้ Runtime → Run all
3. ใช้หน่วยประมวลผลกลาง (CPU) ตามค่าเริ่มต้น ไม่ต้องเปลี่ยนเป็น GPU
4. ไฟล์ที่สร้างขึ้นอยู่ใน `ch08_dimred/outputs/` และหายเมื่อ session จบ ถ้าต้องการเก็บ ให้คัดลอกไปยัง Google Drive

ตัวเลขส่วนใหญ่ควรตรงกับตำรา การคำนวณที่ไวต่อการปัดเศษ (การฝึกโครงข่ายประสาทเทียม
และมอนติคาร์โลในพิกัดต่อเนื่อง) อาจต่างเล็กน้อยเมื่อรันบนเครื่องอื่น ดู `ch08_dimred/README.md`


In [ ]:
# เตรียมโค้ด: ดึงโค้ดจาก GitHub (เมื่อรันบน Colab) และติดตั้งเฉพาะไลบรารีที่ยังไม่มี
import os, sys, subprocess, importlib.util
CHAPTER = 'ch08_dimred'
REPO = 'https://github.com/busarapa-stack/dsml-for-physicists.git'
if 'google.colab' in sys.modules:
    ROOT = '/content/dsml-for-physicists'
    if not os.path.isdir(ROOT):
        subprocess.run(['git', 'clone', '-q', REPO, ROOT], check=True)
else:                                   # Jupyter บนเครื่องตนเอง: เปิดจากโฟลเดอร์ notebooks/
    ROOT = os.path.abspath('..') if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
os.chdir(os.path.join(ROOT, CHAPTER))
os.makedirs('data', exist_ok=True); os.makedirs('outputs', exist_ok=True)
NEED = {'matplotlib': 'matplotlib', 'numpy': 'numpy', 'scipy': 'scipy', 'sklearn': 'scikit-learn>=1.3', 'umap': 'umap-learn>=0.5'}
missing = [req for mod, req in NEED.items() if importlib.util.find_spec(mod) is None]
if missing:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *missing], check=True)
print('โฟลเดอร์ทำงาน:', os.getcwd())
print('ติดตั้งเพิ่ม:', ', '.join(missing) if missing else 'ไม่มี')


## `01_pca_bootstrap.py` — โค้ดที่ 8.1, 8.2

Ch.8  PCA with scikit-learn and bootstrap confidence intervals of the eigenvalues.

In [ ]:
%reset -f
# ทำให้โค้ดทำงานเหมือนสั่ง  python 01_pca_bootstrap.py  จากโฟลเดอร์ของบท
__file__ = '01_pca_bootstrap.py'
import os, sys, time; sys.argv = [__file__]; os.environ['NB_T0'] = str(time.time())
_root = os.path.dirname(os.getcwd())         # ล้างโมดูลและ path ของหัวข้อก่อนหน้า (เช่น _shared.py ของบทอื่น)
sys.path[:] = [os.path.dirname(os.path.abspath(__file__))] + [p for p in sys.path if not os.path.abspath(p or '.').startswith(_root + os.sep)]
for _m in [m for m, v in list(sys.modules.items()) if str(getattr(v, '__file__', '') or '').startswith(_root + os.sep)]:
    del sys.modules[_m]
"""Ch.8  PCA with scikit-learn and bootstrap confidence intervals of the eigenvalues.

Book references: ssec:pca-algo, ssec:ppca-uq, code:pca, code:bootstrap-pca (line for line)
The book leaves the data matrix X of code:pca generic.  Here X is the 21 local AFM features of
code:tsne-umap (3 channels x [value + local mean/std in 3x3, 7x7, 15x15 windows], 16,384 pixels),
built from the SYNTHETIC Chapter 7 map.  The features have different units, so standardising is correct.
"""
import os
from pathlib import Path

import numpy as np
from scipy.ndimage import uniform_filter

HERE = Path(__file__).resolve().parent
os.chdir(HERE)
_s = (HERE.parent / 'ch07_clustering' / '05_afm_three_algorithms.py').read_text()
_s = _s[_s.index('\n# --- code:afm'):]
exec(_s[:_s.index('X_afm, y_afm, glass_afm = make_scaffold_map()')])     # make_scaffold_map() of Chapter 7

_X_afm, _, _ = make_scaffold_map()
_maps = []
for _ch in range(3):
    _m = _X_afm[:, _ch].reshape(128, 128); _maps.append(_m)
    for _w in (3, 7, 15):
        _mu = uniform_filter(_m, _w)
        _maps += [_mu, np.sqrt(np.maximum(uniform_filter(_m * _m, _w) - _mu * _mu, 0))]
X = np.column_stack([_f.ravel() for _f in _maps])        # (16384, 21) in physical units

### โค้ดที่ 8.1

In [ ]:
# --- code:pca ----------------------------------------------------------------
import numpy as np
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

X_scaled = StandardScaler().fit_transform(X)     # X: (n_samples, n_features)
pca = PCA(n_components=10)
X_pca = pca.fit_transform(X_scaled)              # scores on the new axes
print("explained variance ratio:", pca.explained_variance_ratio_)
print("cumulative:", np.cumsum(pca.explained_variance_ratio_))
print("first component:", pca.components_[0])    # loadings on the original features

### โค้ดที่ 8.2

In [ ]:
# --- code:bootstrap-pca ------------------------------------------------------
rng = np.random.default_rng(seed=2026)
n_bootstrap = 200
ev_boot = []
for _ in range(n_bootstrap):
    idx = rng.integers(0, len(X_scaled), size=len(X_scaled))   # resample rows
    ev_boot.append(PCA(n_components=5).fit(X_scaled[idx]).explained_variance_)
ev_boot = np.array(ev_boot)
ci_low, ci_high = np.percentile(ev_boot, [2.5, 97.5], axis=0)

**ส่วนตรวจผล** (ไม่อยู่ในตำรา): พิมพ์ตัวเลขที่ตำรายกมาเพื่อเทียบ

In [ ]:
# --- end of listings ---------------------------------------------------------

if __name__ == '__main__':
    print("\nbootstrap 95 % intervals of the first five eigenvalues (rows are independent pixels here,")
    print("but neighbouring pixels are correlated through the smoothing windows, so these are optimistic):")
    for k, (lo, hi, ev) in enumerate(zip(ci_low, ci_high, pca.explained_variance_[:5]), 1):
        print(f"  PC{k}: {ev:.2f}  [{lo:.2f}, {hi:.2f}]")
    print("Kaiser criterion (lambda > 1 on standardised data) keeps", int((pca.explained_variance_ > 1).sum()), "components")

## `02_ica_cocktail.py` — โค้ดที่ 8.3

Ch.8  Blind source separation: FastICA vs PCA on three mixed signals.

In [ ]:
%reset -f
# ทำให้โค้ดทำงานเหมือนสั่ง  python 02_ica_cocktail.py  จากโฟลเดอร์ของบท
__file__ = '02_ica_cocktail.py'
import os, sys, time; sys.argv = [__file__]; os.environ['NB_T0'] = str(time.time())
_root = os.path.dirname(os.getcwd())         # ล้างโมดูลและ path ของหัวข้อก่อนหน้า (เช่น _shared.py ของบทอื่น)
sys.path[:] = [os.path.dirname(os.path.abspath(__file__))] + [p for p in sys.path if not os.path.abspath(p or '.').startswith(_root + os.sep)]
for _m in [m for m, v in list(sys.modules.items()) if str(getattr(v, '__file__', '') or '').startswith(_root + os.sep)]:
    del sys.modules[_m]
"""Ch.8  Blind source separation: FastICA vs PCA on three mixed signals.

Book references: ssec:ica-algo, code:ica (line for line)
numpy and PCA are imported in code:pca (01_pca_bootstrap.py); they are imported here directly.
"""
import os
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
from sklearn.decomposition import PCA

os.chdir(Path(__file__).resolve().parent)
Path('outputs').mkdir(exist_ok=True)

### โค้ดที่ 8.3

In [ ]:
# --- code:ica ----------------------------------------------------------------
from scipy import signal
from sklearn.decomposition import FastICA

rng = np.random.default_rng(seed=2026)
t = np.linspace(0, 8, 2000)
S_true = np.c_[np.sin(2 * np.pi * 1.3 * t),              # three independent sources
               signal.square(2 * np.pi * 0.7 * t),
               signal.sawtooth(2 * np.pi * 0.45 * t)]
S_true += 0.05 * rng.normal(size=S_true.shape)
A = np.array([[1.0, 0.6, 0.4], [0.5, 1.0, 0.7], [0.3, 0.8, 1.0]])
X_mix = S_true @ A.T                                       # three observed mixtures

ica = FastICA(n_components=3, whiten='unit-variance', random_state=42)
S_ica = ica.fit_transform(X_mix)                           # recovered sources
S_pca = PCA(n_components=3).fit_transform(X_mix)           # for comparison

**ส่วนตรวจผล** (ไม่อยู่ในตำรา): พิมพ์ตัวเลขที่ตำรายกมาเพื่อเทียบ

In [ ]:
# --- end of listings ---------------------------------------------------------

if __name__ == '__main__':
    from scipy.optimize import linear_sum_assignment

    def matched_corr(S, E):
        C = np.abs(np.corrcoef(S.T, E.T)[:3, 3:])
        r, c = linear_sum_assignment(-C)
        return C[r, c]

    print("|correlation| with the true sources (matched):")
    print("  ICA:", matched_corr(S_true, S_ica).round(3))
    print("  PCA:", matched_corr(S_true, S_pca).round(3))
    fig, axes = plt.subplots(3, 3, figsize=(12, 6), sharex=True)
    for j in range(3):
        axes[j, 0].plot(t, S_true[:, j]); axes[j, 1].plot(t, S_ica[:, j]); axes[j, 2].plot(t, S_pca[:, j])
    for ax, title in zip(axes[0], ['true sources', 'FastICA', 'PCA']):
        ax.set_title(title)
    fig.tight_layout(); fig.savefig('outputs/ch08_ica_cocktail.png', dpi=90)
    print("saved outputs/ch08_ica_cocktail.png")

In [ ]:
# แสดงรูปที่ส่วนนี้สร้าง (สคริปต์บันทึกรูปลงไฟล์ใน outputs/)
from IPython.display import Image, display
import glob, os
for _f in sorted(glob.glob('outputs/**/*.png', recursive=True)):
    if os.path.getmtime(_f) >= float(os.environ['NB_T0']):
        print(_f); display(Image(_f))


## `03_tsne_umap_afm.py` — โค้ดที่ 8.4

Ch.8  21 local features of the AFM map, and 2-D views with PCA, t-SNE and UMAP.

In [ ]:
%reset -f
# ทำให้โค้ดทำงานเหมือนสั่ง  python 03_tsne_umap_afm.py  จากโฟลเดอร์ของบท
__file__ = '03_tsne_umap_afm.py'
import os, sys, time; sys.argv = [__file__]; os.environ['NB_T0'] = str(time.time())
_root = os.path.dirname(os.getcwd())         # ล้างโมดูลและ path ของหัวข้อก่อนหน้า (เช่น _shared.py ของบทอื่น)
sys.path[:] = [os.path.dirname(os.path.abspath(__file__))] + [p for p in sys.path if not os.path.abspath(p or '.').startswith(_root + os.sep)]
for _m in [m for m, v in list(sys.modules.items()) if str(getattr(v, '__file__', '') or '').startswith(_root + os.sep)]:
    del sys.modules[_m]
"""Ch.8  21 local features of the AFM map, and 2-D views with PCA, t-SNE and UMAP.

Book references: ssec:umap, code:tsne-umap (line for line), ssec:dr-quality
Needs umap-learn.  make_scaffold_map() is the Chapter 7 function (SYNTHETIC map, seed 2026).
t-SNE and UMAP take about 30-60 s.  UMAP results can differ slightly between machines (numba).
"""
import os
import warnings
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
from sklearn.decomposition import PCA

HERE = Path(__file__).resolve().parent
os.chdir(HERE)
Path('outputs').mkdir(exist_ok=True)
warnings.filterwarnings('ignore')
_s = (HERE.parent / 'ch07_clustering' / '05_afm_three_algorithms.py').read_text()
_s = _s[_s.index('\n# --- code:afm'):]
exec(_s[:_s.index('X_afm, y_afm, glass_afm = make_scaffold_map()')])     # make_scaffold_map() of Chapter 7

### โค้ดที่ 8.4

In [ ]:
# --- code:tsne-umap ----------------------------------------------------------
from scipy.ndimage import uniform_filter
from sklearn.preprocessing import RobustScaler
from sklearn.manifold import TSNE, trustworthiness
import umap                                   # pip install umap-learn

X_afm, y_afm, _ = make_scaffold_map()         # function from the Chapter 7 listing
n = 128
features = []
for ch in range(3):                           # H, S, A
    m = X_afm[:, ch].reshape(n, n)
    features.append(m)
    for w in (3, 7, 15):                      # local mean and local std
        mu = uniform_filter(m, w)
        sd = np.sqrt(np.maximum(uniform_filter(m * m, w) - mu * mu, 0))
        features += [mu, sd]
F = RobustScaler().fit_transform(np.column_stack([f.ravel() for f in features]))

idx = np.random.default_rng(0).choice(len(F), 3000, replace=False)
Z, labels = F[idx], y_afm[idx]
emb = {'PCA':  PCA(n_components=2).fit_transform(Z),
       't-SNE': TSNE(n_components=2, perplexity=30, random_state=42).fit_transform(Z),
       'UMAP': umap.UMAP(n_components=2, n_neighbors=15, min_dist=0.1,
                         random_state=42).fit_transform(Z)}
for name, E in emb.items():
    print(name, "trustworthiness =", round(trustworthiness(Z, E, n_neighbors=10), 3))

**ส่วนตรวจผล** (ไม่อยู่ในตำรา): พิมพ์ตัวเลขที่ตำรายกมาเพื่อเทียบ

In [ ]:
# --- end of listings ---------------------------------------------------------

if __name__ == '__main__':
    from sklearn.model_selection import cross_val_score
    from sklearn.neighbors import KNeighborsClassifier
    cum = np.cumsum(PCA().fit(Z).explained_variance_ratio_)
    print(f"PCA of the 21 features: 2 components {cum[1]:.3f}, 5 components {cum[4]:.3f} of the variance")
    print("5-NN accuracy (5-fold CV) predicting scaffold / substrate:")
    for name, E in list(emb.items()) + [('21 features', Z)]:
        acc = cross_val_score(KNeighborsClassifier(n_neighbors=5), E, labels, cv=5).mean()
        print(f"  {name:12s} {acc:.3f}")
    print(f"  all-scaffold guess {labels.mean():.3f} (this subsample; whole map {y_afm.mean():.3f})")
    _, _, glass = make_scaffold_map(); g = glass[idx]
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
    for ax, (name, E) in zip(axes, emb.items()):
        ax.scatter(*E[labels == 1].T, s=3, c='tab:orange', label='scaffold')
        ax.scatter(*E[(labels == 0) & ~g].T, s=3, c='tab:blue', label='paraffin')
        ax.scatter(*E[g].T, s=3, c='k', label='glass'); ax.set_title(name)
    axes[0].legend(markerscale=4); fig.tight_layout(); fig.savefig('outputs/ch08_afm_embeddings.png', dpi=90)
    print("saved outputs/ch08_afm_embeddings.png")

In [ ]:
# แสดงรูปที่ส่วนนี้สร้าง (สคริปต์บันทึกรูปลงไฟล์ใน outputs/)
from IPython.display import Image, display
import glob, os
for _f in sorted(glob.glob('outputs/**/*.png', recursive=True)):
    if os.path.getmtime(_f) >= float(os.environ['NB_T0']):
        print(_f); display(Image(_f))


## `04_rouse_pca.py` — โค้ดที่ 8.5, 8.6

Ch.8  Rouse chain (exact normal-mode sampling) and PCA of the Cartesian trajectory.

In [ ]:
%reset -f
# ทำให้โค้ดทำงานเหมือนสั่ง  python 04_rouse_pca.py  จากโฟลเดอร์ของบท
__file__ = '04_rouse_pca.py'
import os, sys, time; sys.argv = [__file__]; os.environ['NB_T0'] = str(time.time())
_root = os.path.dirname(os.getcwd())         # ล้างโมดูลและ path ของหัวข้อก่อนหน้า (เช่น _shared.py ของบทอื่น)
sys.path[:] = [os.path.dirname(os.path.abspath(__file__))] + [p for p in sys.path if not os.path.abspath(p or '.').startswith(_root + os.sep)]
for _m in [m for m, v in list(sys.modules.items()) if str(getattr(v, '__file__', '') or '').startswith(_root + os.sep)]:
    del sys.modules[_m]
"""Ch.8  Rouse chain (exact normal-mode sampling) and PCA of the Cartesian trajectory.

Book references: sec:polymer-primary, code:rouse-gen, code:polymer-pca (line for line), ssec:rouse-modes
numpy and PCA are imported in code:pca (01_pca_bootstrap.py); they are imported here directly.
Units k_B T = friction = bond length = 1.
"""
import os
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
from sklearn.decomposition import PCA

os.chdir(Path(__file__).resolve().parent)
Path('outputs').mkdir(exist_ok=True)

### โค้ดที่ 8.5

In [ ]:
# --- code:rouse-gen ----------------------------------------------------------
def make_rouse_trajectory(N=50, n_frames=10000, dt_frame=5.0, seed=2026):
    """Rouse chain (k_B T = friction = bond length = 1) sampled exactly in
    its normal-mode basis; returns Cartesian coordinates (n_frames, 3N)."""
    rng = np.random.default_rng(seed)
    k = 3.0                                              # 3 k_B T / b^2
    i = np.arange(N)
    p = np.arange(1, N)                                  # p = 0 (centre of mass) removed
    Q = np.sqrt(2.0 / N) * np.cos(np.outer(p, np.pi * (i + 0.5) / N))
    a = 4.0 * np.sin(p * np.pi / (2 * N))**2             # Rouse-matrix eigenvalues
    var = 1.0 / (k * a)                                  # <X_p^2> per axis
    tau = 1.0 / (k * a)                                  # relaxation times
    decay = np.exp(-dt_frame / tau)
    Xp = np.empty((n_frames, len(p), 3))
    Xp[0] = rng.normal(size=(len(p), 3)) * np.sqrt(var)[:, None]
    for t in range(1, n_frames):                         # exact Ornstein-Uhlenbeck step
        Xp[t] = (decay[:, None] * Xp[t - 1]
                 + np.sqrt(var * (1 - decay**2))[:, None] * rng.normal(size=(len(p), 3)))
    r = np.einsum('pi,tpc->tic', Q, Xp)                 # bead positions (CM at origin)
    return r.reshape(n_frames, 3 * N), tau

polymer_trajectory, tau_theory = make_rouse_trajectory()

### โค้ดที่ 8.6

In [ ]:
# --- code:polymer-pca --------------------------------------------------------
X = polymer_trajectory                           # (10000, 150), same units in every column
X_centered = X - X.mean(axis=0)                  # no standardisation (see text)

pca = PCA(n_components=12)
X_pca = pca.fit_transform(X_centered)
print("variance ratio:", np.round(pca.explained_variance_ratio_[:9], 3))
print("eigenvalues:  ", np.round(pca.explained_variance_[:9], 1))

N = 50
mode1 = pca.components_[0].reshape(N, 3)         # PC1 as a displacement field
profile = mode1 @ np.linalg.svd(mode1)[2][0]     # project on its dominant direction

**ส่วนตรวจผล** (ไม่อยู่ในตำรา): พิมพ์ตัวเลขที่ตำรายกมาเพื่อเทียบ

In [ ]:
# --- end of listings ---------------------------------------------------------

if __name__ == '__main__':
    p = np.arange(1, N)
    a = 4 * np.sin(p * np.pi / (2 * N))**2
    lam_th = 1 / (3 * a)
    print(f"\ntau_1 = {tau_theory[0]:.1f}; record length {10000 * 5.0:.0f} = {10000 * 5.0 / tau_theory[0]:.0f} tau_1; "
          f"tau_1 = {tau_theory[0] / 5.0:.0f} frames")
    ev = pca.explained_variance_
    for q in range(3):
        print(f"group p={q + 1}: eigenvalues {ev[3*q:3*q+3].round(1)}, mean {ev[3*q:3*q+3].mean():.1f}, theory 1/(3 a_p) = {lam_th[q]:.1f}")
    r = pca.explained_variance_ratio_
    print(f"first 3 components {r[:3].sum():.3f} (theory {(1/a[0]) / (1/a).sum():.3f}; 6/pi^2 = {6/np.pi**2:.3f}); "
          f"first 9 {r[:9].sum():.3f} (theory {(1/a[:3]).sum() / (1/a).sum():.3f})")
    i = np.arange(N)
    print(f"|corr| PC1 profile vs cos(pi(i+1/2)/N) = {abs(np.corrcoef(profile, np.cos(np.pi * (i + .5) / N))[0, 1]):.4f}")
    for j in range(3, 6):
        m = pca.components_[j].reshape(N, 3); pr = m @ np.linalg.svd(m)[2][0]
        print(f"|corr| PC{j + 1} profile vs cos(2 pi(i+1/2)/N) = {abs(np.corrcoef(pr, np.cos(2 * np.pi * (i + .5) / N))[0, 1]):.4f}")
    z = X_pca[:, 0] - X_pca[:, 0].mean()
    acf = np.array([1.0] + [np.mean(z[:-l] * z[l:]) / np.mean(z * z) for l in range(1, 61)])
    for last in (10, 20, 40):                                   # fit ln C(t) on lags 1..last-1 frames
        lag = np.arange(1, last)
        slope = np.polyfit(lag * 5.0, np.log(acf[1:last]), 1)[0]
        print(f"tau_1 from ln C(t), lags up to {5 * (last - 1):.0f} time units: {-1 / slope:.1f} (theory {tau_theory[0]:.1f})")
    tot = X_centered.var(axis=0, ddof=1).sum()
    rg2 = (X_centered.reshape(-1, N, 3)**2).sum(axis=2).mean(axis=1).mean()
    print(f"sum of all variances {tot:.1f} = N <Rg^2> -> <Rg^2> = {tot / N:.2f} (direct {rg2:.2f}); "
          f"Gaussian chain N b^2/6 = {N / 6:.2f}, exact Rouse (N^2-1)/(6N) = {(N**2 - 1) / (6 * N):.2f}")

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    full = PCA().fit(X_centered).explained_variance_
    axes[0].semilogy(np.arange(1, 31), full[:30], 'o', label='PCA'); axes[0].semilogy(np.arange(1, 31), np.repeat(lam_th[:10], 3), 'k_', ms=12, label='1/(3 a_p)'); axes[0].legend()
    axes[0].set_xlabel('component'); axes[0].set_ylabel('eigenvalue'); axes[0].set_title('triplets, ~1/p^2')
    sgn = np.sign(np.corrcoef(profile, np.cos(np.pi * (i + .5) / N))[0, 1])      # the sign of a PC is arbitrary
    axes[1].plot(sgn * profile / np.abs(profile).max(), label='PC1 (sign aligned)'); axes[1].plot(np.cos(np.pi * (i + .5) / N), '--', label='cos p=1')
    axes[1].legend(); axes[1].set_xlabel('bead')
    axes[2].plot(np.arange(61) * 5, acf); axes[2].plot(np.arange(61) * 5, np.exp(-np.arange(61) * 5 / tau_theory[0]), '--')
    axes[2].set_xlabel('t'); axes[2].set_title('C(t) of PC1 vs exp(-t/tau_1)')
    fig.tight_layout(); fig.savefig('outputs/ch08_rouse_pca.png', dpi=90)
    print("saved outputs/ch08_rouse_pca.png")

In [ ]:
# แสดงรูปที่ส่วนนี้สร้าง (สคริปต์บันทึกรูปลงไฟล์ใน outputs/)
from IPython.display import Image, display
import glob, os
for _f in sorted(glob.glob('outputs/**/*.png', recursive=True)):
    if os.path.getmtime(_f) >= float(os.environ['NB_T0']):
        print(_f); display(Image(_f))


## `05_eigenvalue_ci_block.py`

Ch.8  Confidence interval of the mean of the first three eigenvalues of the Rouse trajectory:

In [ ]:
%reset -f
# ทำให้โค้ดทำงานเหมือนสั่ง  python 05_eigenvalue_ci_block.py  จากโฟลเดอร์ของบท
__file__ = '05_eigenvalue_ci_block.py'
import os, sys, time; sys.argv = [__file__]; os.environ['NB_T0'] = str(time.time())
_root = os.path.dirname(os.getcwd())         # ล้างโมดูลและ path ของหัวข้อก่อนหน้า (เช่น _shared.py ของบทอื่น)
sys.path[:] = [os.path.dirname(os.path.abspath(__file__))] + [p for p in sys.path if not os.path.abspath(p or '.').startswith(_root + os.sep)]
for _m in [m for m, v in list(sys.modules.items()) if str(getattr(v, '__file__', '') or '').startswith(_root + os.sep)]:
    del sys.modules[_m]
"""Ch.8  Confidence interval of the mean of the first three eigenvalues of the Rouse trajectory:
independent-frame bootstrap vs block bootstrap.

Book references: ssec:dr-ci, E8.5, review question 7
"""
import os
from pathlib import Path

HERE = Path(__file__).resolve().parent
os.chdir(HERE)
_s = (HERE / '04_rouse_pca.py').read_text()
exec(_s[_s.index('import numpy'):_s.index('\n# --- end of listings')].replace("os.chdir(Path(__file__).resolve().parent)", ""))

SEED = 2026
B = 200


def mean_top3(Z):
    return PCA(n_components=3).fit(Z).explained_variance_.mean()


print(f"mean of the first three eigenvalues: {mean_top3(X_centered):.2f} (theory {1 / (12 * np.sin(np.pi / 100)**2):.2f})")
rng = np.random.default_rng(SEED)
iid = [mean_top3(X_centered[rng.integers(0, len(X_centered), len(X_centered))]) for _ in range(B)]
lo, hi = np.percentile(iid, [2.5, 97.5])
print(f"independent-frame bootstrap 95 % CI: [{lo:.1f}, {hi:.1f}]  width {hi - lo:.1f}")
for L in (100, 200, 500):
    nb = len(X_centered) // L
    rng = np.random.default_rng(SEED)
    blk = []
    for _ in range(B):
        start = rng.integers(0, nb, nb)
        blk.append(mean_top3(X_centered[(start[:, None] * L + np.arange(L)).ravel()]))
    blo, bhi = np.percentile(blk, [2.5, 97.5])
    print(f"block bootstrap, blocks of {L} frames: [{blo:.1f}, {bhi:.1f}]  width {bhi - blo:.1f} "
          f"({(bhi - blo) / (hi - lo):.1f} x wider)")

# แนวคำตอบของแบบฝึกหัดที่ต้องรันโค้ด
ควรลองทำเองก่อน แล้วจึงรันเซลล์เหล่านี้เพื่อเทียบคำตอบ

## `exercises/E8_1_curse_embedded.py`

E8.1  Max/min pairwise distance ratio for uniform data in D = 3, 10, 50, 100, and for a 2-D plane embedded in 100-D.

In [ ]:
%reset -f
# ทำให้โค้ดทำงานเหมือนสั่ง  python exercises/E8_1_curse_embedded.py  จากโฟลเดอร์ของบท
__file__ = 'exercises/E8_1_curse_embedded.py'
import os, sys, time; sys.argv = [__file__]; os.environ['NB_T0'] = str(time.time())
_root = os.path.dirname(os.getcwd())         # ล้างโมดูลและ path ของหัวข้อก่อนหน้า (เช่น _shared.py ของบทอื่น)
sys.path[:] = [os.path.dirname(os.path.abspath(__file__))] + [p for p in sys.path if not os.path.abspath(p or '.').startswith(_root + os.sep)]
for _m in [m for m, v in list(sys.modules.items()) if str(getattr(v, '__file__', '') or '').startswith(_root + os.sep)]:
    del sys.modules[_m]
"""E8.1  Max/min pairwise distance ratio for uniform data in D = 3, 10, 50, 100, and for a 2-D plane embedded in 100-D.
The exercise fixes no seed; five seeds are shown because the ratio depends on the single closest pair."""
import numpy as np
from scipy.spatial.distance import pdist

for seed in (2026, 0, 1, 2, 42):
    rng = np.random.default_rng(seed)
    r = {D: pdist(rng.random((200, D))) for D in (3, 10, 50, 100)}
    P = rng.random((200, 2))                          # 2-D points
    E = P @ rng.random((2, 100))                      # embedded in 100-D by a random 2 x 100 matrix
    d2, de = pdist(P), pdist(E)
    print(f"seed {seed:4d}: " + "  ".join(f"D={D}: {d.max() / d.min():6.1f}" for D, d in r.items())
          + f"  | plane in 100-D: {de.max() / de.min():7.0f}  (same points before embedding: {d2.max() / d2.min():6.0f})")

## `exercises/E8_2_harmonic_chain.py`

E8.2  PCA of thermal fluctuations of a fixed-end harmonic chain (N = 20, m = k = k_B T = 1) vs analytic normal modes.

In [ ]:
%reset -f
# ทำให้โค้ดทำงานเหมือนสั่ง  python exercises/E8_2_harmonic_chain.py  จากโฟลเดอร์ของบท
__file__ = 'exercises/E8_2_harmonic_chain.py'
import os, sys, time; sys.argv = [__file__]; os.environ['NB_T0'] = str(time.time())
_root = os.path.dirname(os.getcwd())         # ล้างโมดูลและ path ของหัวข้อก่อนหน้า (เช่น _shared.py ของบทอื่น)
sys.path[:] = [os.path.dirname(os.path.abspath(__file__))] + [p for p in sys.path if not os.path.abspath(p or '.').startswith(_root + os.sep)]
for _m in [m for m, v in list(sys.modules.items()) if str(getattr(v, '__file__', '') or '').startswith(_root + os.sep)]:
    del sys.modules[_m]
"""E8.2  PCA of thermal fluctuations of a fixed-end harmonic chain (N = 20, m = k = k_B T = 1) vs analytic normal modes."""
import numpy as np
from sklearn.decomposition import PCA

N = 20
K = 2 * np.eye(N) - np.eye(N, k=1) - np.eye(N, k=-1)
n = np.arange(1, N + 1)
j = np.arange(1, 6)
u = np.array([np.sqrt(2 / (N + 1)) * np.sin(jj * np.pi * n / (N + 1)) for jj in j])
theory = 1 / (4 * np.sin(j * np.pi / (2 * (N + 1)))**2)
print("theory 1/omega^2:", theory.round(1))
for seed in (2026, 0, 1):
    rng = np.random.default_rng(seed)
    Q = rng.multivariate_normal(np.zeros(N), np.linalg.inv(K), size=5000)
    pca = PCA().fit(Q)
    print(f"seed {seed:4d}: |<PC_j, u_j>| = {np.abs((pca.components_[:5] * u).sum(axis=1)).round(3)}  "
          f"eigenvalues {pca.explained_variance_[:5].round(1)}  PC1 share {pca.explained_variance_ratio_[0]:.3f}")
print(f"relative sampling error of an eigenvalue ~ sqrt(2/5000) = {np.sqrt(2 / 5000):.3f}")

## `exercises/E8_3_spectral_unmixing.py`

E8.3  Spectral unmixing of 40 mixtures of three pure spectra: FastICA vs PCA vs NMF.

In [ ]:
%reset -f
# ทำให้โค้ดทำงานเหมือนสั่ง  python exercises/E8_3_spectral_unmixing.py  จากโฟลเดอร์ของบท
__file__ = 'exercises/E8_3_spectral_unmixing.py'
import os, sys, time; sys.argv = [__file__]; os.environ['NB_T0'] = str(time.time())
_root = os.path.dirname(os.getcwd())         # ล้างโมดูลและ path ของหัวข้อก่อนหน้า (เช่น _shared.py ของบทอื่น)
sys.path[:] = [os.path.dirname(os.path.abspath(__file__))] + [p for p in sys.path if not os.path.abspath(p or '.').startswith(_root + os.sep)]
for _m in [m for m, v in list(sys.modules.items()) if str(getattr(v, '__file__', '') or '').startswith(_root + os.sep)]:
    del sys.modules[_m]
"""E8.3  Spectral unmixing of 40 mixtures of three pure spectra: FastICA vs PCA vs NMF.

Pure spectra (Gaussian peaks on 500 points, 400-800 nm):
  s1 = peak 480 nm (width 12) + 0.3 x peak 620 nm (15);  s2 = 530 nm (20) + 0.5 x 700 nm (12);  s3 = 590 nm (25).
Concentrations from rng.dirichlet (they sum to one), noise sd 0.01.  Ten data sets (seeds 0-9).
The run with concentrations multiplied by a random total amount (0.5-1.5) shows the effect of closure.
"""
import warnings

import numpy as np
from scipy.optimize import linear_sum_assignment
from sklearn.decomposition import NMF, PCA, FastICA

warnings.filterwarnings('ignore')
wl = np.linspace(400, 800, 500)
G = lambda c, w: np.exp(-0.5 * ((wl - c) / w)**2)
S = np.array([G(480, 12) + 0.3 * G(620, 15), G(530, 20) + 0.5 * G(700, 12), G(590, 25)])
print("correlation between the pure spectra:", np.corrcoef(S)[np.triu_indices(3, 1)].round(2))


def matched(E):
    C = np.abs(np.corrcoef(S, E)[:3, 3:])
    r, c = linear_sum_assignment(-C)
    return np.sort(C[r, c])


for closure in (True, False):
    res = {'FastICA': [], 'PCA': [], 'NMF': []}; share3 = []
    for seed in range(10):
        rng = np.random.default_rng(seed)
        conc = rng.dirichlet(np.ones(3), size=40)
        if not closure:
            conc *= rng.uniform(0.5, 1.5, size=(40, 1))
        X = conc @ S + 0.01 * rng.normal(size=(40, 500))
        share3.append(PCA(3).fit(X).explained_variance_ratio_[2])
        res['FastICA'].append(matched(FastICA(3, whiten='unit-variance', random_state=42, max_iter=2000).fit_transform(X.T).T))
        res['PCA'].append(matched(PCA(3).fit(X).components_))
        res['NMF'].append(matched(NMF(3, init='nndsvda', max_iter=2000, random_state=42).fit(np.clip(X, 0, None)).components_))
    print(f"\nconcentrations {'sum to one (Dirichlet, as in the exercise)' if closure else 'x random total amount'}: "
          f"3rd PCA component carries {100 * np.mean(share3):.2f} % of the variance")
    for k, v in res.items():
        v = np.array(v)
        print(f"  {k:8s} |corr| range {v.min():.3f}-{v.max():.3f}; median per component (worst..best) {np.median(v, axis=0).round(3)}")

rng = np.random.default_rng(0)
X = rng.dirichlet(np.ones(3), size=40) @ S + 0.01 * rng.normal(size=(40, 500))
for name, E in [('FastICA', FastICA(3, whiten='unit-variance', random_state=42, max_iter=2000).fit_transform(X.T).T),
                ('PCA', PCA(3).fit(X).components_)]:
    E = E * np.sign(E[np.arange(3), np.abs(E).argmax(axis=1)])[:, None]     # make the largest excursion positive
    print(f"{name}: most negative value / largest value of each component (seed 0): {(E.min(axis=1) / E.max(axis=1)).round(2)}")

## `exercises/E8_4_embedding_params.py`

E8.4  PCA / t-SNE / UMAP of the 21 AFM features: trustworthiness, 5-NN accuracy, perplexity and n_neighbors,

In [ ]:
%reset -f
# ทำให้โค้ดทำงานเหมือนสั่ง  python exercises/E8_4_embedding_params.py  จากโฟลเดอร์ของบท
__file__ = 'exercises/E8_4_embedding_params.py'
import os, sys, time; sys.argv = [__file__]; os.environ['NB_T0'] = str(time.time())
_root = os.path.dirname(os.getcwd())         # ล้างโมดูลและ path ของหัวข้อก่อนหน้า (เช่น _shared.py ของบทอื่น)
sys.path[:] = [os.path.dirname(os.path.abspath(__file__))] + [p for p in sys.path if not os.path.abspath(p or '.').startswith(_root + os.sep)]
for _m in [m for m, v in list(sys.modules.items()) if str(getattr(v, '__file__', '') or '').startswith(_root + os.sep)]:
    del sys.modules[_m]
"""E8.4  PCA / t-SNE / UMAP of the 21 AFM features: trustworthiness, 5-NN accuracy, perplexity and n_neighbors,
and K-Means (k = 3) MCC with 21 features vs the three raw channels.  About 3-4 minutes."""
import os
import warnings
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

HERE = Path(__file__).resolve().parent.parent
os.chdir(HERE)
warnings.filterwarnings('ignore')
__file__ = str(HERE / '03_tsne_umap_afm.py')
_s = (HERE / '03_tsne_umap_afm.py').read_text()
exec(_s[:_s.index('\n# --- end of listings')].split('"""', 2)[2])      # builds F, Z, labels, emb

from sklearn.cluster import KMeans
from sklearn.metrics import matthews_corrcoef
from sklearn.model_selection import cross_val_score
from sklearn.neighbors import KNeighborsClassifier

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
_, _, glass = make_scaffold_map(); gl = glass[idx]
runs = [('t-SNE', p, TSNE(n_components=2, perplexity=p, random_state=42)) for p in (5, 30, 100)] + \
       [('UMAP', k, umap.UMAP(n_components=2, n_neighbors=k, min_dist=0.1, random_state=42)) for k in (5, 15, 100)]
for ax, (name, par, model) in zip(axes.ravel(), runs):
    E = model.fit_transform(Z)
    acc = cross_val_score(KNeighborsClassifier(n_neighbors=5), E, labels, cv=5).mean()
    print(f"{name:5s} {'perplexity' if name == 't-SNE' else 'n_neighbors'}={par:3d}: trustworthiness "
          f"{trustworthiness(Z, E, n_neighbors=10):.3f}, 5-NN accuracy {acc:.3f}")
    ax.scatter(*E[labels == 1].T, s=2, c='tab:orange'); ax.scatter(*E[(labels == 0) & ~gl].T, s=2, c='tab:blue')
    ax.scatter(*E[gl].T, s=2, c='k'); ax.set_title(f"{name} {par}")
fig.tight_layout(); fig.savefig('outputs/E8_4_params.png', dpi=80)
print("saved outputs/E8_4_params.png")


def mcc_wta(lab):
    share = {c: y_afm[lab == c].mean() for c in np.unique(lab)}
    return matthews_corrcoef(y_afm, (lab == max(share, key=share.get)).astype(int))


print(f"K-Means k=3 MCC (winner-take-all): 21 features {mcc_wta(KMeans(3, n_init=20, random_state=42).fit_predict(F)):.3f}, "
      f"3 channels {mcc_wta(KMeans(3, n_init=20, random_state=42).fit_predict(RobustScaler().fit_transform(X_afm))):.3f}")
acc21 = cross_val_score(KNeighborsClassifier(n_neighbors=5), Z, labels, cv=5).mean()
acc3 = cross_val_score(KNeighborsClassifier(n_neighbors=5), RobustScaler().fit_transform(X_afm)[idx], labels, cv=5).mean()
print(f"5-NN accuracy on the same 3,000 pixels: 21 features {acc21:.3f}, 3 channels {acc3:.3f}")

In [ ]:
# แสดงรูปที่ส่วนนี้สร้าง (สคริปต์บันทึกรูปลงไฟล์ใน outputs/)
from IPython.display import Image, display
import glob, os
for _f in sorted(glob.glob('outputs/**/*.png', recursive=True)):
    if os.path.getmtime(_f) >= float(os.environ['NB_T0']):
        print(_f); display(Image(_f))


## `exercises/E8_5_rouse_modes.py`

E8.5  Rouse chain: eigenvalue triplets, mode shapes for p = 1, 2, tau_1, <Rg^2>, and the block-bootstrap CI.

In [ ]:
%reset -f
# ทำให้โค้ดทำงานเหมือนสั่ง  python exercises/E8_5_rouse_modes.py  จากโฟลเดอร์ของบท
__file__ = 'exercises/E8_5_rouse_modes.py'
import os, sys, time; sys.argv = [__file__]; os.environ['NB_T0'] = str(time.time())
_root = os.path.dirname(os.getcwd())         # ล้างโมดูลและ path ของหัวข้อก่อนหน้า (เช่น _shared.py ของบทอื่น)
sys.path[:] = [os.path.dirname(os.path.abspath(__file__))] + [p for p in sys.path if not os.path.abspath(p or '.').startswith(_root + os.sep)]
for _m in [m for m, v in list(sys.modules.items()) if str(getattr(v, '__file__', '') or '').startswith(_root + os.sep)]:
    del sys.modules[_m]
"""E8.5  Rouse chain: eigenvalue triplets, mode shapes for p = 1, 2, tau_1, <Rg^2>, and the block-bootstrap CI.
Most numbers come from 04_rouse_pca.py and 05_eigenvalue_ci_block.py, which this script runs."""
import os
import subprocess
import sys
from pathlib import Path

HERE = Path(__file__).resolve().parent.parent
os.chdir(HERE)
for f in ('04_rouse_pca.py', '05_eigenvalue_ci_block.py'):
    print(f"--- {f}")
    subprocess.run([sys.executable, f], check=True)

## `exercises/E8_6_dr_quality.py`

E8.6  Quality of 2-D embeddings of the Rouse trajectory (2,000 random frames): trustworthiness and continuity (k = 10),

In [ ]:
%reset -f
# ทำให้โค้ดทำงานเหมือนสั่ง  python exercises/E8_6_dr_quality.py  จากโฟลเดอร์ของบท
__file__ = 'exercises/E8_6_dr_quality.py'
import os, sys, time; sys.argv = [__file__]; os.environ['NB_T0'] = str(time.time())
_root = os.path.dirname(os.getcwd())         # ล้างโมดูลและ path ของหัวข้อก่อนหน้า (เช่น _shared.py ของบทอื่น)
sys.path[:] = [os.path.dirname(os.path.abspath(__file__))] + [p for p in sys.path if not os.path.abspath(p or '.').startswith(_root + os.sep)]
for _m in [m for m, v in list(sys.modules.items()) if str(getattr(v, '__file__', '') or '').startswith(_root + os.sep)]:
    del sys.modules[_m]
"""E8.6  Quality of 2-D embeddings of the Rouse trajectory (2,000 random frames): trustworthiness and continuity (k = 10),
PCA reconstruction error for d = 3, 9, 30, and linear regression of R_ee on the 2-D coordinates.  About 1-2 minutes."""
import os
import warnings
from pathlib import Path

HERE = Path(__file__).resolve().parent.parent
os.chdir(HERE)
warnings.filterwarnings('ignore')
_s = (HERE / '04_rouse_pca.py').read_text()
exec(_s[_s.index('import numpy'):_s.index('\n# --- end of listings')].replace("os.chdir(Path(__file__).resolve().parent)", ""))

import umap
from sklearn.linear_model import LinearRegression
from sklearn.manifold import TSNE, trustworthiness
from sklearn.model_selection import cross_val_score

rng = np.random.default_rng(2026)
sel = rng.choice(len(X_centered), 2000, replace=False)
Xs = X_centered[sel]
r = polymer_trajectory[sel].reshape(-1, N, 3)
ree = np.linalg.norm(r[:, -1] - r[:, 0], axis=1)

tot = ((Xs - Xs.mean(0))**2).sum()
for d in (2, 3, 9, 30):
    p = PCA(d).fit(Xs)
    rec = p.inverse_transform(p.transform(Xs))
    print(f"PCA d={d:2d}: relative reconstruction error {((Xs - rec)**2).sum() / tot:.3f}")

for seed in (42, 0):
    emb = {'PCA': PCA(2).fit_transform(Xs),
           't-SNE': TSNE(2, perplexity=30, random_state=seed).fit_transform(Xs),
           'UMAP': umap.UMAP(n_components=2, n_neighbors=15, min_dist=0.1, random_state=seed).fit_transform(Xs)}
    for name, E in emb.items():
        tw = trustworthiness(Xs, E, n_neighbors=10)
        co = trustworthiness(E, Xs, n_neighbors=10)            # continuity: roles swapped
        r2 = cross_val_score(LinearRegression(), E, ree, cv=5, scoring='r2').mean()
        print(f"seed {seed:2d} {name:5s}: trustworthiness {tw:.3f}  continuity {co:.3f}  CV R^2 of R_ee from 2-D coords {r2:.3f}")
from sklearn.neighbors import KNeighborsRegressor
cv = lambda model, feats, target: cross_val_score(model, feats, target, cv=5, scoring='r2').mean()
print("R_ee is a length: flipping the sign of a mode amplitude leaves it unchanged, so it is an even function of the scores.")
for name, E in emb.items():                                   # seed 0 embeddings from the loop above
    print(f"{name:5s} 2-D: CV R^2 of R_ee  linear {cv(LinearRegression(), E, ree):.3f}, "
          f"linear on (coords, coords^2) {cv(LinearRegression(), np.c_[E, E**2], ree):.3f}, 15-NN {cv(KNeighborsRegressor(15), E, ree):.3f}")
Rvec = r[:, -1] - r[:, 0]
for d in (3, 9, 30):
    Zd = PCA(d).fit_transform(Xs)
    print(f"PCA {d:2d} components: R_ee from squared scores {cv(LinearRegression(), Zd**2, ree):.3f}; "
          f"x-component of the R_ee vector, linear in the scores {cv(LinearRegression(), Zd, Rvec[:, 0]):.3f}")
# R_ee in terms of modes: R_ee vector = sum_p X_p (psi_p(N) - psi_p(1)), non-zero only for odd p
pp = np.arange(1, N)
w = np.sqrt(2 / N) * (np.cos(pp * np.pi * (N - 0.5) / N) - np.cos(pp * np.pi * 0.5 / N))
contrib = 3 * w**2 / (3 * 4 * np.sin(pp * np.pi / (2 * N))**2)      # 3 axes x w_p^2 <X_p^2>
print("share of <R_ee^2> from modes p = 1..6:", (contrib[:6] / contrib.sum()).round(3), "(even modes contribute zero);",
      f"theory <R_ee^2> = {contrib.sum():.1f} = (N-1) b^2 = {N - 1}; sample {np.mean(ree**2):.1f}")